# 02 — Engenharia de Features Anti-Leakage, Experimentos e Baseline Heurístico
**Hipótese Científica:** Features longitudinais construídas estritamente no instante $t$ (com particionamento temporal *out-of-time* $t \rightarrow t+1$) superam uma regra heurística simples na identificação preventiva de necessidade de acompanhamento, sem vazamento de informação futura nem superajuste (*overfitting*) à janela histórica.

In [ ]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.pipeline.features import build_longitudinal_dataset
from src.pipeline.train import run_training_pipeline

splits, feature_cols = build_longitudinal_dataset()
print(f"Quantidade de features de entrada: {len(feature_cols)}")
print("Features:", feature_cols)

## 1. Integridade Temporal dos Splits e Taxa-Base do Alvo ($t+1$)
Auditoria do volume de amostras e da prevalência da classe positiva (`target_acompanhamento_t1`) em treino, validação e teste fora do tempo.

In [ ]:
resumo_splits = []
for nome_split, df_split in splits.items():
    target = df_split["target_acompanhamento_t1"]
    resumo_splits.append({
        "split": nome_split,
        "linhas": len(df_split),
        "colunas": df_split.shape[1],
        "positivos": int(target.sum()),
        "taxa_positiva": round(float(target.mean()), 4),
        "nulos_em_features": int(df_split[feature_cols].isna().sum().sum()),
    })
display(pd.DataFrame(resumo_splits))

## 2. Verificação Formal de Anti-Leakage e Esquema
Garantia de que variáveis de desfecho, identificadores diretos ou atributos exclusivos de $t+1$ não contaminam o vetor de features usado no treino e na inferência.

In [ ]:
termos_suspeitos = ("target", "t1", "t_plus_1", "futuro", "label")
vazamentos = [c for c in feature_cols if any(t in c.lower() for t in termos_suspeitos)]
assert not vazamentos, f"Possível leakage detectado em feature_cols: {vazamentos}"
for nome_split, df_split in splits.items():
    faltantes = [c for c in feature_cols if c not in df_split.columns]
    assert not faltantes, f"Colunas ausentes em {nome_split}: {faltantes}"
print("Auditoria anti-leakage e consistência de esquema aprovadas em todos os splits.")

## 3. Execução do Pipeline de Treino e Comparação com Baseline
Acionamento de `run_training_pipeline()` para treinar os candidatos, avaliar frente ao baseline heurístico e registar métricas/artefatos.

In [ ]:
summary = run_training_pipeline()
if isinstance(summary, dict):
    display(pd.json_normalize(summary, sep=".").T.rename(columns={0: "valor"}))
else:
    print("Resumo do treino:", summary)

## 4. Rigor Científico: Métricas, Overfitting e Limitações
* **Escolha de Métricas:** Em cenários com desbalanceamento de classes e custo assimétrico de falso negativo (deixar de sinalizar um estudante que precisará de acompanhamento), *PR-AUC*, *ROC-AUC*, *Recall* e *F2-Score* são mais informativos que acurácia global.
* **Risco de Overfitting e Domain Shift:** Modelos baseados em árvores podem superajustar padrões específicos de 2022–2023 se não forem validados em janelas posteriores (*out-of-time*) e calibrados quanto ao limiar de decisão.
* **Estratégia de Validação:** Divisão cronológica estrita com verificação de estabilidade temporal antes de qualquer promoção via *Quality Gate*.